# Neural reduced GP

AutoEncoder / VAE / supervised AutoEncoder を軽量設定で構築し、非線形 latent representation の使い方を確認します。


## このNotebookで学ぶこと

- AE・VAE・教師ありAEによる非線形入力次元削減の違い
- **事前学習型**（reducerを構築時に学習し、GPを後からfit）と**Joint学習型**（encoderとGPを同時最適化）の違い
- 元の6次元入力でPosteriorを取得し、予測平均・不確実性を可視化する方法
- robotorchanの既存モデルとBoTorchのqLogEIによる候補提案・評価
- Mixed・MultiTask・Kroneckerの入力・出力構造と候補選択の制約

**方針：** 獲得関数による探索を実践例の中心とし、複雑なモデルでは既存APIに適合する範囲で使用します。

## データ


In [ ]:
import torch
from robotorchan.models import AutoEncoderGP, VAEGP, SupervisedAutoEncoderGP

torch.manual_seed(0)
torch.set_default_dtype(torch.double)
X = torch.rand(20, 6)
Y = torch.sin(4 * X[:, :1]) + X[:, 1:2]


### AE・VAE・教師ありAEの違い

| Reducer | 表現学習の目的 | 注意点 |
|---|---|---|
| AutoEncoder（AE） | 入力Xを復元する潜在表現を学習 | 目的Yに重要な方向を必ず残すわけではない |
| Variational AutoEncoder（VAE） | 再構成に加え潜在分布を正則化 | KL重みや学習データ量に影響される |
| Supervised AE | Yの予測情報も表現学習に利用 | 少量データでは教師信号への過適合に注意 |

この例では6次元入力を2次元の潜在表現へ縮約します。ただし、候補探索は元の6次元入力空間で行います。

## 軽量なreducer学習


In [ ]:
models = {
    "AE": AutoEncoderGP(X, Y, latent_dim=2, hidden_dims=(8,), epochs=3),
    "VAE": VAEGP(X, Y, latent_dim=2, hidden_dims=(8,), epochs=3),
    "SupervisedAE": SupervisedAutoEncoderGP(X, Y, latent_dim=2, hidden_dims=(8,), epochs=3),
}
for name, model in models.items():
    print(name, model.reduced_input_dim)


## Posterior


In [ ]:
X_test = torch.rand(2, 6)
for name, model in models.items():
    print(name, model.posterior(X_test).mean.squeeze(-1))


## Model fitting

Reducerは構築時に学習済みですが、GP部分は既存の`make_mll()`で別途学習します。


### 学習手順の確認

ここまでの `epochs=3` はreducerの軽量な事前学習設定です。**GPのハイパーパラメータ学習はまだ完了していません。** 次のセルで各モデルの `fit()` を呼び、内部で既存の `make_mll()` に基づくBoTorchのMLL最適化を利用します。

Joint系は後半で `model.fit()` または `training_loss()` とPyTorch optimizerを使います。学習手順が異なる理由は、EncoderのパラメータもGPと同時に更新するためです。

In [ ]:

for name, model in models.items():
    model.fit()
    model.eval()
    print(name, "GP fitted")


### 予測平均と潜在関数の事後不確実性を可視化

6次元のうち第1変数だけを動かし、残りを0.5に固定した断面を表示します。帯は潜在関数の事後標準偏差の約1.96倍です。観測ノイズを含む区間ではありません。

軽量な3 epoch学習では精度を保証しません。これは**予測APIと不確実性の読み方**を確認するための図です。

In [ ]:
import matplotlib.pyplot as plt

slice_X = torch.full((121, 6), 0.5)
slice_X[:, 0] = torch.linspace(0.0, 1.0, len(slice_X))
true_slice = torch.sin(4 * slice_X[:, 0]) + slice_X[:, 1]
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5), sharey=True)
for ax, (name, surrogate) in zip(axes, models.items()):
    surrogate.eval()
    with torch.no_grad():
        post = surrogate.posterior(slice_X)
        mean = post.mean.squeeze(-1)
        std = post.variance.clamp_min(0).sqrt().squeeze(-1)
    ax.plot(slice_X[:, 0], true_slice, "--", label="true slice")
    ax.plot(slice_X[:, 0], mean, label="posterior mean")
    ax.fill_between(slice_X[:, 0], mean - 1.96 * std,
                    mean + 1.96 * std, alpha=0.2,
                    label="approx. 95% latent interval")
    ax.set_title(name)
    ax.set_xlabel("x[0] (others fixed at 0.5)")
axes[0].set_ylabel("objective")
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

## Acquisition-based candidate generation and evaluation

各モデルについて元の6次元空間でqLogEIを最適化し、提案点を目的関数で評価します。


In [ ]:
from botorch.acquisition.logei import qLogExpectedImprovement
from botorch.optim import optimize_acqf
from botorch.sampling.normal import SobolQMCNormalSampler


def objective(X):
    return torch.sin(4 * X[..., :1]) + X[..., 1:2]


bounds = torch.stack([torch.zeros(6), torch.ones(6)])
for name, model in models.items():
    acqf = qLogExpectedImprovement(
        model=model,
        best_f=Y.max(),
        sampler=SobolQMCNormalSampler(sample_shape=torch.Size([32])),
    )
    candidate, value = optimize_acqf(
        acq_function=acqf,
        bounds=bounds,
        q=1,
        num_restarts=2,
        raw_samples=32,
    )
    assert candidate.shape == torch.Size([1, 6])
    print(name, "candidate:", candidate)
    print(name, "acquisition:", value)
    print(name, "objective:", objective(candidate))


### qLogEIと最適化の役割

`qLogExpectedImprovement` は改善の期待値を使う獲得関数です。`optimize_acqf()` は、その獲得関数が高くなる候補を元の6次元入力空間で探します。これは単純な事後平均最大化とは異なり、予測分布の不確実性も利用します。

候補が得られたら `objective(candidate)` で合成目的関数を評価します。通常の実験では、この部分を実測値の取得に置き換えます。

## Jointly trained neural encoders

`JointEncoderGP` と `HybridAutoEncoderGP` は、reducerを事前学習して固定するモデルとは異なり、GPの損失とencoderを同時に最適化します。Hybridでは再構成損失も含むため、既存の `training_loss()` を用います。


In [ ]:
from robotorchan.models import HybridAutoEncoderGP, JointEncoderGP

joint_models = {
    "JointEncoderGP": JointEncoderGP(X, Y, latent_dim=2, hidden_dims=(8,)),
    "HybridAutoEncoderGP": HybridAutoEncoderGP(
        X, Y, latent_dim=2, hidden_dims=(8,), reconstruction_weight=0.1
    ),
}
for name, model in joint_models.items():
    model.fit(epochs=5, lr=0.01)
    with torch.no_grad():
        pool = torch.rand(32, X.shape[-1])
        scores = model.posterior(pool).mean.reshape(-1)
    candidate = pool[scores.argmax()].unsqueeze(0)
    assert candidate.shape == (1, X.shape[-1])
    print(name, "candidate:", candidate)
    print(name, "objective:", objective(candidate))


### Joint学習型での候補提案

Joint Encoder系は、EncoderとGPを一緒に更新します。学習済みのJointモデルでも `posterior(X)` がBoTorchの獲得関数に必要な分布を返す場合は、qLogEIで候補を順位付けできます。

次の例では**元の6次元空間の有限候補集合**に対してqLogEIを計算します。連続空間を `optimize_acqf()` で最適化する例とは異なりますが、事後平均だけで候補を選ぶよりもBOの探索方針に近い例です。

## Joint VAE and mixed joint encoder models

VAEではGPの予測損失に再構成・KL正則化を加えます。Mixedモデルは連続特徴量だけをencoderに通し、カテゴリ特徴量を保持します。


In [ ]:
from robotorchan.models import JointVAEGP, MixedJointEncoderGP, MixedJointVAEGP

mixed_X = torch.cat(
    [X, torch.randint(0, 2, (X.shape[0], 1)).to(X)], dim=-1
)
mixed_pool = torch.cat(
    [torch.rand(32, X.shape[-1]), torch.randint(0, 2, (32, 1)).to(X)],
    dim=-1,
)
additional_joint_models = {
    "JointVAEGP": (JointVAEGP(
        X, Y, latent_dim=2, hidden_dims=(8,), beta=0.1,
        reconstruction_weight=0.1,
    ), torch.rand(32, X.shape[-1])),
    "MixedJointEncoderGP": (MixedJointEncoderGP(
        mixed_X, Y, latent_dim=2, cat_dims=[6], hidden_dims=(8,),
    ), mixed_pool),
    "MixedJointVAEGP": (MixedJointVAEGP(
        mixed_X, Y, latent_dim=2, cat_dims=[6], hidden_dims=(8,),
        beta=0.1, reconstruction_weight=0.1,
    ), mixed_pool),
}
for name, (model, pool) in additional_joint_models.items():
    model.fit(epochs=5, lr=0.01)
    with torch.no_grad():
        scores = model.posterior(pool).mean.reshape(-1)
    candidate = pool[scores.argmax()].unsqueeze(0)
    assert candidate.shape[-1] == pool.shape[-1]
    if pool.shape[-1] == 7:
        assert candidate[0, -1].item() in (0, 1)
    print(name, "candidate:", candidate)
    print(name, "objective:", objective(candidate[..., :6]))


### Jointモデル：qLogEIによる候補プール選択

前のセルで学習済みのJointモデルについて、同じ候補プールをqLogEIで評価します。既存の事後平均による候補と比較できるよう、両方を残します。

In [ ]:
joint_pool = torch.rand(64, X.shape[-1])
for name, joint_model in joint_models.items():
    joint_model.eval()
    joint_acqf = qLogExpectedImprovement(
        model=joint_model,
        best_f=Y.max(),
        sampler=SobolQMCNormalSampler(sample_shape=torch.Size([32])),
    )
    with torch.no_grad():
        joint_acq_values = joint_acqf(joint_pool.unsqueeze(-2)).reshape(-1)
    selected = joint_acq_values.argmax().item()
    proposed = joint_pool[selected:selected + 1]
    assert proposed.shape == (1, 6)
    print(name, "qLogEI pool candidate:", proposed)
    print(name, "acquisition:", joint_acq_values[selected])
    print(name, "evaluated objective:", objective(proposed))

## Frozen mixed neural input reducers

Mixed AutoEncoderとMixed VAEでは連続入力だけを事前学習したreducerで変換し、カテゴリ変数は保持します。GP部分は既存の `make_mll()` を使って学習します。


In [ ]:
from robotorchan.models import MixedAutoEncoderGP, MixedVAEGP

frozen_mixed_models = {
    "MixedAutoEncoderGP": MixedAutoEncoderGP(
        mixed_X, Y, latent_dim=2, cat_dims=[6], hidden_dims=(8,), epochs=3
    ),
    "MixedVAEGP": MixedVAEGP(
        mixed_X, Y, latent_dim=2, cat_dims=[6], hidden_dims=(8,), epochs=3
    ),
}
for name, model in frozen_mixed_models.items():
    model.fit()
    model.eval()
    with torch.no_grad():
        scores = model.posterior(mixed_pool).mean.reshape(-1)
    candidate = mixed_pool[scores.argmax()].unsqueeze(0)
    assert candidate.shape == (1, 7)
    assert candidate[0, -1].item() in (0, 1)
    print(name, "candidate:", candidate)
    print(name, "objective:", objective(candidate[..., :6]))


### Mixedモデルの探索空間

Mixedモデルでは連続入力6列とカテゴリ入力1列を分けて扱います。有限候補プールを使用する場合もカテゴリ列が有効な値であることを確認します。

BoTorchの `optimize_acqf_mixed()` を利用する選択肢もありますが、ここでは全Jointモデルへの機械的な適用は避けます。各モデルのPosterior・勾配・入力制約との適合を確認してから選択する必要があります。

## Supervised mixed neural reducers

教師ありAutoEncoder・VAEは目的変数もreducer学習に使用します。連続入力だけを削減し、カテゴリ入力は保持します。


In [ ]:
from robotorchan.models import MixedSupervisedAutoEncoderGP, MixedSupervisedVAEGP

supervised_mixed_models = {
    "MixedSupervisedAutoEncoderGP": MixedSupervisedAutoEncoderGP(
        mixed_X, Y, latent_dim=2, cat_dims=[6], hidden_dims=(8,),
        epochs=3, supervised_weight=0.5,
    ),
    "MixedSupervisedVAEGP": MixedSupervisedVAEGP(
        mixed_X, Y, latent_dim=2, cat_dims=[6], hidden_dims=(8,),
        epochs=3, supervised_weight=0.5, beta=0.1,
    ),
}
for name, model in supervised_mixed_models.items():
    model.fit()
    model.eval()
    with torch.no_grad():
        scores = model.posterior(mixed_pool).mean.reshape(-1)
    candidate = mixed_pool[scores.argmax()].unsqueeze(0)
    assert candidate.shape == (1, 7)
    assert candidate[0, -1].item() in (0, 1)
    print(name, "candidate:", candidate)
    print(name, "objective:", objective(candidate[..., :6]))


## Frozen neural reduction for long-format multitask GPs

2タスクのlong-formatデータを用意し、タスクIDを除いた連続入力のみをAE・VAE・教師ありAEで削減します。タスク0の候補を元の入力空間で提案します。


In [ ]:
from robotorchan.models import (
    AutoEncoderMultiTaskGP,
    SupervisedAutoEncoderMultiTaskGP,
    VAEMultiTaskGP,
)

task_X = torch.cat(
    [
        torch.cat([X, torch.zeros(X.shape[0], 1)], dim=-1),
        torch.cat([X, torch.ones(X.shape[0], 1)], dim=-1),
    ],
    dim=0,
)
task_Y = torch.cat([Y, Y + 0.2], dim=0)
multitask_neural_models = {
    "AutoEncoderMultiTaskGP": AutoEncoderMultiTaskGP(
        task_X, task_Y, task_feature=6, latent_dim=2, hidden_dims=(8,), epochs=3
    ),
    "VAEMultiTaskGP": VAEMultiTaskGP(
        task_X, task_Y, task_feature=6, latent_dim=2, hidden_dims=(8,), epochs=3
    ),
    "SupervisedAutoEncoderMultiTaskGP": SupervisedAutoEncoderMultiTaskGP(
        task_X, task_Y, task_feature=6, latent_dim=2, hidden_dims=(8,), epochs=3
    ),
}
task_pool = torch.cat([torch.rand(32, 6), torch.zeros(32, 1)], dim=-1)
for name, model in multitask_neural_models.items():
    model.fit()
    model.eval()
    with torch.no_grad():
        scores = model.posterior(task_pool).mean.reshape(-1)
    candidate = task_pool[scores.argmax()].unsqueeze(0)
    assert candidate.shape == (1, 7)
    assert candidate[0, -1].item() == 0
    print(name, "candidate:", candidate)
    print(name, "task 0 objective:", objective(candidate[..., :6]))


### MultiTaskとKroneckerの候補選択

long-format MultiTaskは入力の末尾にタスクIDを持ち、Kronecker型は複数タスクが出力列に並びます。どちらも対象タスクを明示して候補を評価する必要があります。

後続の例では**タスク0の事後平均**による有限候補選択を維持しています。獲得関数が利用できないと判定したわけではなく、対象タスクの固定やPosteriorの出力選択を伴う最適化を、このNotebookでは一律には追加していません。

## Supervised VAE multitask and frozen Kronecker multitask

教師ありVAEのlong-formatモデルと、AutoEncoder/VAEのblock-designモデルを構築します。Kroneckerモデルは2出力Posteriorの第1タスクで候補を順位付けします。


In [ ]:
from robotorchan.models import (
    AutoEncoderKroneckerMultiTaskGP,
    SupervisedVAEMultiTaskGP,
    VAEKroneckerMultiTaskGP,
)

supervised_vae_multitask = SupervisedVAEMultiTaskGP(
    task_X, task_Y, task_feature=6, latent_dim=2,
    hidden_dims=(8,), epochs=3, beta=0.1, supervised_weight=0.5,
)
supervised_vae_multitask.fit()
supervised_vae_multitask.eval()
with torch.no_grad():
    task_scores = supervised_vae_multitask.posterior(task_pool).mean.reshape(-1)
task_candidate = task_pool[task_scores.argmax()].unsqueeze(0)
print("SupervisedVAEMultiTaskGP candidate:", task_candidate)
print("Task 0 objective:", objective(task_candidate[..., :6]))

block_Y = torch.cat([Y, Y + 0.2], dim=-1)
kronecker_neural_models = {
    "AutoEncoderKroneckerMultiTaskGP": AutoEncoderKroneckerMultiTaskGP(
        X, block_Y, latent_dim=2, hidden_dims=(8,), epochs=3
    ),
    "VAEKroneckerMultiTaskGP": VAEKroneckerMultiTaskGP(
        X, block_Y, latent_dim=2, hidden_dims=(8,), epochs=3
    ),
}
block_pool = torch.rand(32, X.shape[-1])
for name, model in kronecker_neural_models.items():
    model.fit()
    model.eval()
    with torch.no_grad():
        posterior = model.posterior(block_pool)
        assert posterior.mean.shape == (32, 2)
        scores = posterior.mean[:, 0]
    candidate = block_pool[scores.argmax()].unsqueeze(0)
    print(name, "candidate:", candidate)
    print(name, "task 0 objective:", objective(candidate))


## Supervised neural Kronecker multitask GPs

教師ありAutoEncoder/VAEで複数タスクの出力を考慮した表現を学習し、Kronecker GPを学習します。候補提案にはタスク0の予測平均を使用します。


In [ ]:
from robotorchan.models import (
    SupervisedAutoEncoderKroneckerMultiTaskGP,
    SupervisedVAEKroneckerMultiTaskGP,
)

supervised_kronecker_models = {
    "SupervisedAutoEncoderKroneckerMultiTaskGP": (
        SupervisedAutoEncoderKroneckerMultiTaskGP(
            X, block_Y, latent_dim=2, hidden_dims=(8,),
            epochs=3, supervised_weight=0.5,
        )
    ),
    "SupervisedVAEKroneckerMultiTaskGP": SupervisedVAEKroneckerMultiTaskGP(
        X, block_Y, latent_dim=2, hidden_dims=(8,),
        epochs=3, beta=0.1, supervised_weight=0.5,
    ),
}
for name, model in supervised_kronecker_models.items():
    model.fit()
    model.eval()
    with torch.no_grad():
        posterior = model.posterior(block_pool)
        assert posterior.mean.shape == (32, 2)
        scores = posterior.mean[:, 0]
    candidate = block_pool[scores.argmax()].unsqueeze(0)
    print(name, "candidate:", candidate)
    print(name, "task 0 objective:", objective(candidate))


## Joint neural multitask models

JointEncoder系はEncoderとGPを `training_loss()` で同時学習します。Long-formatとblock-designをそれぞれ元の入力空間で評価します。


In [ ]:
from robotorchan.models import (
    JointEncoderKroneckerMultiTaskGP,
    JointEncoderMultiTaskGP,
)

joint_multitask_models = {
    "JointEncoderMultiTaskGP": (
        JointEncoderMultiTaskGP(
            task_X, task_Y, task_feature=6, latent_dim=2, hidden_dims=(8,)
        ),
        task_pool,
    ),
    "JointEncoderKroneckerMultiTaskGP": (
        JointEncoderKroneckerMultiTaskGP(
            X, block_Y, latent_dim=2, hidden_dims=(8,)
        ),
        block_pool,
    ),
}
for name, (model, pool) in joint_multitask_models.items():
    model.fit(epochs=5, lr=0.01)
    with torch.no_grad():
        posterior_mean = model.posterior(pool).mean
        scores = (
            posterior_mean[:, 0]
            if "Kronecker" in name
            else posterior_mean.reshape(-1)
        )
    candidate = pool[scores.argmax()].unsqueeze(0)
    print(name, "candidate:", candidate)
    print(name, "task 0 objective:", objective(candidate[..., :6]))


## Hybrid AutoEncoder multitask models

再構成損失を含むJoint Encoderをlong-formatとKroneckerの両形式で学習し、元の入力空間の候補を評価します。


In [ ]:
from robotorchan.models import (
    HybridAutoEncoderKroneckerMultiTaskGP,
    HybridAutoEncoderMultiTaskGP,
)

hybrid_multitask_models = {
    "HybridAutoEncoderMultiTaskGP": (
        HybridAutoEncoderMultiTaskGP(
            task_X, task_Y, task_feature=6, latent_dim=2,
            hidden_dims=(8,), reconstruction_weight=0.1,
        ),
        task_pool,
    ),
    "HybridAutoEncoderKroneckerMultiTaskGP": (
        HybridAutoEncoderKroneckerMultiTaskGP(
            X, block_Y, latent_dim=2, hidden_dims=(8,),
            reconstruction_weight=0.1,
        ),
        block_pool,
    ),
}
for name, (model, pool) in hybrid_multitask_models.items():
    model.fit(epochs=5, lr=0.01)
    with torch.no_grad():
        posterior_mean = model.posterior(pool).mean
        scores = (
            posterior_mean[:, 0]
            if "Kronecker" in name
            else posterior_mean.reshape(-1)
        )
    candidate = pool[scores.argmax()].unsqueeze(0)
    print(name, "candidate:", candidate)
    print(name, "task 0 objective:", objective(candidate[..., :6]))


## Joint VAE multitask models

Joint VAEではGP損失・再構成損失・KL損失を同時に最適化します。Long-formatとKroneckerの両形式でタスク0の候補を提案します。


In [ ]:
from robotorchan.models import (
    JointVAEKroneckerMultiTaskGP,
    JointVAEMultiTaskGP,
)

joint_vae_multitask_models = {
    "JointVAEMultiTaskGP": (
        JointVAEMultiTaskGP(
            task_X, task_Y, task_feature=6, latent_dim=2,
            hidden_dims=(8,), reconstruction_weight=0.1, beta=0.1,
        ),
        task_pool,
    ),
    "JointVAEKroneckerMultiTaskGP": (
        JointVAEKroneckerMultiTaskGP(
            X, block_Y, latent_dim=2, hidden_dims=(8,),
            reconstruction_weight=0.1, beta=0.1,
        ),
        block_pool,
    ),
}
for name, (model, pool) in joint_vae_multitask_models.items():
    model.fit(epochs=5, lr=0.01)
    with torch.no_grad():
        posterior_mean = model.posterior(pool).mean
        scores = (
            posterior_mean[:, 0]
            if "Kronecker" in name
            else posterior_mean.reshape(-1)
        )
    candidate = pool[scores.argmax()].unsqueeze(0)
    print(name, "candidate:", candidate)
    print(name, "task 0 objective:", objective(candidate[..., :6]))


## Remaining neural reduction model families

教師ありVAE、Mixed Hybrid AutoEncoder、Mixed Joint Encoder MultiTaskの学習から候補評価までを確認します。候補は有限プールの予測平均最大点であり、獲得関数の最適化ではありません。


In [ ]:
from robotorchan.models import (
    MixedHybridAutoEncoderGP,
    MixedJointEncoderMultiTaskGP,
    SupervisedVAEGP,
)

supervised_vae = SupervisedVAEGP(
    X, Y, latent_dim=2, hidden_dims=(8,), epochs=3, beta=0.1
)
supervised_vae.fit()
supervised_vae.eval()
with torch.no_grad():
    scores = supervised_vae.posterior(block_pool).mean.reshape(-1)
candidate = block_pool[scores.argmax()].unsqueeze(0)
print("SupervisedVAEGP candidate:", candidate)
print("Objective:", objective(candidate))

mixed_hybrid = MixedHybridAutoEncoderGP(
    mixed_X, Y, latent_dim=2, cat_dims=[6],
    hidden_dims=(8,), reconstruction_weight=0.1,
)
mixed_task_X = torch.cat(
    [
        torch.cat([mixed_X, torch.zeros(mixed_X.shape[0], 1)], dim=-1),
        torch.cat([mixed_X, torch.ones(mixed_X.shape[0], 1)], dim=-1),
    ],
    dim=0,
)
mixed_task_pool = torch.cat(
    [mixed_pool, torch.zeros(mixed_pool.shape[0], 1)], dim=-1
)
mixed_joint_multitask = MixedJointEncoderMultiTaskGP(
    mixed_task_X, task_Y, task_feature=7, latent_dim=2,
    cat_dims=[6], hidden_dims=(8,),
)
for name, model, pool in [
    ("MixedHybridAutoEncoderGP", mixed_hybrid, mixed_pool),
    ("MixedJointEncoderMultiTaskGP", mixed_joint_multitask, mixed_task_pool),
]:
    model.fit(epochs=5, lr=0.01)
    with torch.no_grad():
        scores = model.posterior(pool).mean.reshape(-1)
    candidate = pool[scores.argmax()].unsqueeze(0)
    print(name, "candidate:", candidate)
    print(name, "objective:", objective(candidate[..., :6]))


## まとめ：モデル・学習・探索の対応

| モデル群 | 学習 | 候補提案 |
|---|---|---|
| AE / VAE / Supervised AE | `model.fit()` | qLogEI + `optimize_acqf()` |
| JointEncoderGP / HybridAutoEncoderGP | `model.fit()` | 事後平均プール選択 + qLogEIプール選択 |
| Joint VAE / Mixed Joint | `model.fit(epochs=5, lr=0.01)` | 事後平均プール選択 |
| Frozen Mixed | `model.fit()` | 事後平均プール選択 |
| MultiTask / Kronecker | モデルごとの既存学習API | 対象タスクの事後平均プール選択 |

このNotebookでは、通常のニューラル縮約GPとJoint単一出力モデルで獲得関数による探索を示し、他のモデルは既存の入力・出力契約を尊重します。学習epoch数は説明用の軽量設定であり、精度・最適化性能の比較実験ではありません。